# Stereo-duplicate analysis and clean-up

The combined dataset is keyed on **canonical isomeric SMILES**. Sources record stereochemistry inconsistently (EGNN never records it; PATENTS and Bradley record it for some compounds), so the same connectivity can appear more than once: once without stereo marks and once with them. This notebook quantifies how many compounds are affected and how much their melting points differ (Part 1), and then applies the agreed clean-up rule (Part 2).

Two SMILES forms are kept side by side:

- `SMILES_isomeric`: the stored canonical isomeric SMILES (stereo marks `@`, `/`, `\` kept);
- `SMILES_nonisomeric`: RDKit canonical SMILES with `isomericSmiles=False` (stereo marks removed).

Compounds sharing a `SMILES_nonisomeric` form a **stereo group**. A compound is **specified** if its isomeric SMILES carries stereo marks and **unspecified** otherwise.

| Class | Definition | Interpretation |
|---|---|---|
| **A** | group of 1 unspecified + 1 specified | most likely the same measurement, with stereo dropped by one source |
| **B** | group of 1 unspecified + ≥2 specified | ambiguous: the unspecified entry could be either isomer, or the racemate |
| **C** | group of ≥2 specified, no unspecified | genuine stereoisomers, whose MPs may legitimately differ |
| **D** | specified, no group partner | unique stereo-labelled compound; not a duplicate |
| **Unique, no stereo marks** | unspecified, no group partner | either cannot have stereoisomers, or could but no source reported stereo; not a duplicate |

It reads `outputs/combined_data_before_stereo_cleanup.parquet`, written by `1_combine_data.ipynb`. Part 1 writes no files. Part 2 writes to `outputs/`:

- `combined_data.parquet`: the final dataset (one entry per non-isomeric SMILES);
- `stereo_cleanup_audit.csv`: every compound affected by the clean-up and the action taken;
- `combined_data_graph_embeddings.csv.gz`: the precomputed graph embeddings restricted to the compounds of `combined_data.parquet`.

In [1]:
from itertools import combinations
from pathlib import Path
import gzip

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')

data_dir_candidates = [Path.cwd(), Path.cwd() / '0_data', Path.cwd().parent / '0_data']
DATA_DIR = next((path.resolve() for path in data_dir_candidates
                 if (path / 'outputs' / 'combined_data_before_stereo_cleanup.parquet').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate 0_data/outputs/combined_data_before_stereo_cleanup.parquet. Run '
        '1_combine_data.ipynb first, then run this notebook '
        'from the repository root or from the 0_data directory.'
    )
BEFORE_CLEANUP_PATH = DATA_DIR / 'outputs' / 'combined_data_before_stereo_cleanup.parquet'
OUTPUT_DATA = DATA_DIR / 'outputs' / 'combined_data.parquet'
OUTPUT_STEREO_AUDIT = DATA_DIR / 'outputs' / 'stereo_cleanup_audit.csv'
GRAPH_EMBEDDINGS_SOURCE = (DATA_DIR / 'input_datasets' / 'graph_embeddings'
                           / 'combined_data_SingleViews_graph_embeddings.csv.gz')
OUTPUT_GRAPH_EMBEDDINGS = DATA_DIR / 'outputs' / 'combined_data_graph_embeddings.csv.gz'

SOURCE_NAMES = {1: 'ChemXplore', 2: 'EGNN', 3: 'PATENTS', 4: 'Bradley'}
HIGH_THRESHOLD = 250.0
STEREO_MARKS = r'[@/\\]'
STEREO_CONFLICT_TOLERANCE_C = 10.0
CATEGORIES = ['A: 1 unspecified + 1 specified',
              'B: 1 unspecified + ≥2 specified',
              'C: specified only']
TOLERANCES = [0, 2, 5, 10]

pd.set_option('display.max_colwidth', 90)
print(f'Dataset analysed: {BEFORE_CLEANUP_PATH.relative_to(DATA_DIR.parent)}')

Dataset analysed: 0_data/outputs/combined_data_before_stereo_cleanup.parquet


# Part 1: stereo-duplicate analysis

## Load the dataset and derive both SMILES forms

`Source_label` lists every source behind a compound's consensus MP (for example `2+3` = EGNN and PATENTS).

In [2]:
combined_data = pd.read_parquet(
    BEFORE_CLEANUP_PATH, columns=['SMILES', 'MP', 'Source', 'MP_quality']
).rename(columns={'SMILES': 'SMILES_isomeric'})

def source_label(sources):
    return '+'.join(str(int(source)) for source in sorted(sources))

def source_names(label):
    return ' + '.join(SOURCE_NAMES[int(source)] for source in label.split('+'))

def nonisomeric_smiles(smiles):
    return Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=False)

combined_data['Source_label'] = combined_data['Source'].map(source_label)
combined_data['SMILES_nonisomeric'] = combined_data['SMILES_isomeric'].map(nonisomeric_smiles)
combined_data['Has_stereo'] = combined_data['SMILES_isomeric'].str.contains(STEREO_MARKS)
has_chiral_mark = combined_data['SMILES_isomeric'].str.contains('@')
has_bond_mark = combined_data['SMILES_isomeric'].str.contains(r'[/\\]')
combined_data['Stereo_element'] = np.select(
    [has_chiral_mark & has_bond_mark, has_chiral_mark, has_bond_mark],
    ['chiral centre + double bond', 'chiral centre', 'double bond'], 'none',
)
group_size = combined_data.groupby('SMILES_nonisomeric')['SMILES_isomeric'].transform('size')
combined_data['In_stereo_group'] = group_size.gt(1)

if combined_data['SMILES_isomeric'].duplicated().any():
    raise ValueError('The dataset should contain one row per isomeric SMILES.')

display(combined_data.loc[combined_data['Has_stereo'],
                          ['SMILES_isomeric', 'SMILES_nonisomeric', 'MP', 'Source_label']].head())

,SMILES_isomeric,SMILES_nonisomeric,MP,Source_label
3,Br/C(=C/c1ccccc1)c1ccc(Br)c2c(Br)cccc12,BrC(=Cc1ccccc1)c1ccc(Br)c2c(Br)cccc12,171.0,4
4,Br/C(=N/Nc1ccccc1)c1ccccc1,BrC(=NNc1ccccc1)c1ccccc1,190.0,4
5,Br/C=C/c1ccccc1,BrC=Cc1ccccc1,7.0,3
7,BrC(=C(\Br)c1ccccc1)/C(Br)=C(\Br)c1ccccc1,BrC(C(Br)=C(Br)c1ccccc1)=C(Br)c1ccccc1,236.0,3+4
39,BrC/C(=C(\CBr)c1ccccc1)c1ccccc1,BrCC(=C(CBr)c1ccccc1)c1ccccc1,98.0,3


## Isomeric vs non-isomeric breakdown

Every compound is one row (one isomeric SMILES). Compounds without stereo marks and without a group partner are further split with `Chem.FindPotentialStereo`: some have no possible stereocentre or stereo double bond and therefore cannot have stereoisomers; the others could exist as stereoisomers, but no source reported stereo for them.

The number of unique non-isomeric SMILES is smaller than the number of compounds because each stereo group collapses to a single non-isomeric SMILES.

In [3]:
def has_potential_stereo(smiles):
    return len(Chem.FindPotentialStereo(Chem.MolFromSmiles(smiles))) > 0

labelled = combined_data['Has_stereo']
grouped = combined_data['In_stereo_group']
unique_unlabelled = ~labelled & ~grouped
stereo_possible = combined_data.loc[unique_unlabelled, 'SMILES_isomeric'].map(has_potential_stereo)

n_compounds = len(combined_data)
indent = ' ' * 6
breakdown = pd.DataFrame([
    ('All compounds (unique isomeric SMILES)', n_compounds),
    (f'{indent}With stereo marks', int(labelled.sum())),
    (f'{indent * 2}in a stereo group (specified members of A/B/C)', int((labelled & grouped).sum())),
    (f'{indent * 2}unique: category D', int((labelled & ~grouped).sum())),
    (f'{indent}Without stereo marks', int((~labelled).sum())),
    (f'{indent * 2}in a stereo group (unspecified members of A/B)', int((~labelled & grouped).sum())),
    (f'{indent * 2}unique', int(unique_unlabelled.sum())),
    (f'{indent * 3}no possible stereo element (cannot have stereoisomers)', int((~stereo_possible).sum())),
    (f'{indent * 3}possible stereo element, none reported', int(stereo_possible.sum())),
], columns=['Compounds', 'Count'])
breakdown['% of compounds'] = 100 * breakdown['Count'] / n_compounds
display(breakdown.style.format({'Count': '{:,}', '% of compounds': '{:.2f}'})
        .set_properties(subset=['Compounds'], **{'text-align': 'left'}).hide(axis='index'))

n_groups = combined_data.loc[grouped, 'SMILES_nonisomeric'].nunique()
n_unique_nonisomeric = combined_data['SMILES_nonisomeric'].nunique()
reconciliation = pd.DataFrame([
    ('Stereo groups (each group counted once)', n_groups),
    ('Category D (unique, with stereo marks)', int((labelled & ~grouped).sum())),
    ('Unique, without stereo marks', int(unique_unlabelled.sum())),
    ('Unique non-isomeric SMILES', n_unique_nonisomeric),
], columns=['Non-isomeric SMILES', 'Count'])
assert reconciliation['Count'].iloc[:3].sum() == n_unique_nonisomeric
display(reconciliation.style.format({'Count': '{:,}'})
        .set_properties(subset=['Non-isomeric SMILES'], **{'text-align': 'left'}).hide(axis='index'))
print(f'Compounds that would collapse if stereo were ignored: '
      f'{n_compounds:,} − {n_unique_nonisomeric:,} = {n_compounds - n_unique_nonisomeric:,}')

Compounds,Count,% of compounds
All compounds (unique isomeric SMILES),"304,508",100.00
With stereo marks,"31,801",10.44
in a stereo group (specified members of A/B/C),"27,065",8.89
unique: category D,"4,736",1.56
Without stereo marks,"272,707",89.56
in a stereo group (unspecified members of A/B),"26,662",8.76
unique,"246,045",80.80
no possible stereo element (cannot have stereoisomers),"173,929",57.12
"possible stereo element, none reported","72,116",23.68


Non-isomeric SMILES,Count
Stereo groups (each group counted once),"26,728"
"Category D (unique, with stereo marks)","4,736"
"Unique, without stereo marks","246,045"
Unique non-isomeric SMILES,"277,509"


Compounds that would collapse if stereo were ignored: 304,508 − 277,509 = 26,999


## Isotope check

`isomericSmiles=False` removes isotope labels as well as stereo marks, so a labelled compound (for example a `[13C]` analogue) could in principle fall into the same group as its unlabelled parent. That would not be a stereo duplicate, and labelled compounds can have different MPs.

The check removes only stereochemistry (`Chem.RemoveStereochemistry`, isotopes kept) and requires every member of a group to give the same structure. Deuterium and tritium are written as explicit atoms (`[H]`) in non-isomeric SMILES, so D/T compounds cannot collide with their parents; the assert guards against other labels in future data.

In [4]:
def stereo_stripped_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol)

members = combined_data.loc[grouped].copy()
members['SMILES_stereo_stripped'] = members['SMILES_isomeric'].map(stereo_stripped_smiles)
structures_per_group = members.groupby('SMILES_nonisomeric')['SMILES_stereo_stripped'].nunique()
isotope_labelled = combined_data['SMILES_isomeric'].str.contains(r'\[\d+[A-Z]')

print(f'Isotope-labelled compounds in the dataset: {int(isotope_labelled.sum())}')
print(f'Isotope-labelled compounds inside stereo groups: {int((isotope_labelled & grouped).sum())}')
print(f'Groups whose members differ by more than stereochemistry: {int(structures_per_group.gt(1).sum())}')
assert structures_per_group.eq(1).all(), 'Some groups differ by isotopes or other non-stereo features.'

Isotope-labelled compounds in the dataset: 23
Isotope-labelled compounds inside stereo groups: 0
Groups whose members differ by more than stereochemistry: 0


## Category table

Every pair of members within a group is compared (a category-B group of 1 + 2 members gives three pairs), using the absolute MP difference |ΔMP| in °C. A pair *crosses 250 °C* when one member is low MP (<250 °C) and the other high MP (≥250 °C), so the compound's low/high class depends on which entry is kept. Pair columns are blank for classes that are not groups.

In [5]:
group_table = members.groupby('SMILES_nonisomeric').agg(
    N_members=('SMILES_isomeric', 'size'), N_specified=('Has_stereo', 'sum'),
)
group_table['N_unspecified'] = group_table['N_members'] - group_table['N_specified']
group_table['Category'] = np.select(
    [group_table['N_unspecified'].eq(1) & group_table['N_specified'].eq(1),
     group_table['N_unspecified'].eq(1) & group_table['N_specified'].ge(2),
     group_table['N_unspecified'].eq(0)],
    CATEGORIES, 'other',
)
assert group_table['Category'].ne('other').all(), 'Unexpected group composition.'
members = members.join(group_table[['Category']], on='SMILES_nonisomeric')

def pair_rows(group):
    rows = []
    for first, second in combinations(group.itertuples(index=False), 2):
        if first.Has_stereo and not second.Has_stereo:
            first, second = second, first
        rows.append({
            'Category': first.Category,
            'Pair_type': 'unspecified–specified' if not first.Has_stereo else 'specified–specified',
            'SMILES_a': first.SMILES_isomeric, 'MP_a': first.MP,
            'Source_a': first.Source_label, 'Quality_a': first.MP_quality,
            'SMILES_b': second.SMILES_isomeric, 'MP_b': second.MP,
            'Source_b': second.Source_label, 'Quality_b': second.MP_quality,
        })
    return rows

pair_columns = ['SMILES_nonisomeric', 'Category', 'Has_stereo', 'SMILES_isomeric',
                'MP', 'Source_label', 'MP_quality']
pairs = pd.DataFrame([
    row for _, group in members[pair_columns].groupby('SMILES_nonisomeric', sort=False)
    for row in pair_rows(group)
])
pairs['Abs_dMP'] = (pairs['MP_a'] - pairs['MP_b']).abs()
pairs['Crosses_250'] = (pairs[['MP_a', 'MP_b']].min(axis=1).lt(HIGH_THRESHOLD)
                        & pairs[['MP_a', 'MP_b']].max(axis=1).ge(HIGH_THRESHOLD))

def agreement_stats(frame):
    return pd.Series({
        'Pairs': len(frame),
        '% identical MP': 100 * frame['Abs_dMP'].eq(0).mean(),
        '% ≤2 °C': 100 * frame['Abs_dMP'].le(2).mean(),
        'Pairs >10 °C': int(frame['Abs_dMP'].gt(10).sum()),
        'Pairs crossing 250 °C': int(frame['Crosses_250'].sum()),
    })

group_counts = group_table.groupby('Category').agg(
    Groups=('N_members', 'size'), Compounds=('N_members', 'sum'),
)
pair_stats = pairs.groupby('Category').apply(agreement_stats, include_groups=False)
category_table = group_counts.join(pair_stats).reindex(CATEGORIES)
category_table.loc['D: specified, unique', 'Compounds'] = int((labelled & ~grouped).sum())
category_table.loc['Unique, no stereo marks: cannot have stereoisomers', 'Compounds'] = int((~stereo_possible).sum())
category_table.loc['Unique, no stereo marks: stereoisomers possible, none reported', 'Compounds'] = int(stereo_possible.sum())
assert category_table['Compounds'].sum() == n_compounds, 'Classes do not cover the dataset.'
category_table.loc['Total'] = category_table.sum(min_count=1)
category_table.loc['Total', ['% identical MP', '% ≤2 °C']] = np.nan
category_table.insert(2, '% of dataset', 100 * category_table['Compounds'] / n_compounds)

count_format = '{:,.0f}'
display(category_table.rename_axis('Class').style.format({
    'Groups': count_format, 'Compounds': count_format, '% of dataset': '{:.2f}',
    'Pairs': count_format, '% identical MP': '{:.1f}', '% ≤2 °C': '{:.1f}',
    'Pairs >10 °C': count_format, 'Pairs crossing 250 °C': count_format,
}, na_rep=''))

category_b = pairs.loc[pairs['Category'].eq(CATEGORIES[1])]
for pair_type, frame in category_b.groupby('Pair_type'):
    stats = agreement_stats(frame)
    print(f'Category B, {pair_type} pairs: {stats["Pairs"]:,.0f} pairs, '
          f'{stats["% identical MP"]:.1f}% identical, {stats["% ≤2 °C"]:.1f}% within 2 °C')

,Groups,Compounds,% of dataset,Pairs,% identical MP,% ≤2 °C,Pairs >10 °C,Pairs crossing 250 °C
Class,,,,,,,,
A: 1 unspecified + 1 specified,"26,412","52,824",17.35,"26,412",97.1,99.5,59,7
B: 1 unspecified + ≥2 specified,250,767,0.25,806,44.4,74.3,117,26
C: specified only,66,136,0.04,75,34.7,70.7,21,1
"D: specified, unique",,"4,736",1.56,,,,,
"Unique, no stereo marks: cannot have stereoisomers",,"173,929",57.12,,,,,
"Unique, no stereo marks: stereoisomers possible, none reported",,"72,116",23.68,,,,,
Total,"26,728","304,508",100.00,"27,293",,,197,34


Category B, specified–specified pairs: 289 pairs, 33.6% identical, 68.9% within 2 °C
Category B, unspecified–specified pairs: 517 pairs, 50.5% identical, 77.4% within 2 °C


## Source evidence for category A

The unspecified member's sources are compared with the specified member's. `EGNN → PATENTS` means the unspecified entry came only from EGNN and the specified entry only from PATENTS, i.e. the EGNN copy of a PATENTS record.

In [6]:
category_a = pairs.loc[pairs['Category'].eq(CATEGORIES[0])].copy()
category_a['Pairing'] = np.select(
    [category_a['Source_a'].eq('2') & category_a['Source_b'].eq('3'),
     category_a['Source_a'].str.contains('2')],
    ['EGNN → PATENTS', 'Unspecified entry includes EGNN (other pairings)'],
    'Unspecified entry without EGNN',
)
source_evidence = category_a.groupby('Pairing').apply(agreement_stats, include_groups=False)
source_evidence.insert(1, '% of category A', 100 * source_evidence['Pairs'] / len(category_a))
source_evidence = source_evidence.drop(columns='Pairs crossing 250 °C').sort_values('Pairs', ascending=False)
display(source_evidence.style.format({
    'Pairs': count_format, '% of category A': '{:.1f}', '% identical MP': '{:.1f}',
    '% ≤2 °C': '{:.1f}', 'Pairs >10 °C': count_format,
}))

,Pairs,% of category A,% identical MP,% ≤2 °C,Pairs >10 °C
Pairing,,,,,
EGNN → PATENTS,"24,533",92.9,99.7,99.9,9
Unspecified entry includes EGNN (other pairings),"1,673",6.3,62.0,94.6,36
Unspecified entry without EGNN,206,0.8,67.0,86.9,14


## Merge-tolerance counts (category A)

No rule is applied here. A category-A pair whose |ΔMP| is within a tolerance τ could be merged into a single entry (for example, keeping the stereo-specified SMILES). Pairs outside τ, and all of categories B and C, would remain separate compounds and need a grouped split (all members of a stereo group in the same train/validation/test split).

In [7]:
merge_rows = []
for tolerance in TOLERANCES:
    mergeable = int(category_a['Abs_dMP'].le(tolerance).sum())
    merge_rows.append({
        'Tolerance τ (°C)': tolerance,
        'Pairs mergeable': mergeable,
        '% of category A': 100 * mergeable / len(category_a),
        'Pairs kept separate': len(category_a) - mergeable,
        'Compounds after merging': n_compounds - mergeable,
    })
display(pd.DataFrame(merge_rows).style.format({
    'Pairs mergeable': '{:,}', '% of category A': '{:.1f}',
    'Pairs kept separate': '{:,}', 'Compounds after merging': '{:,}',
}).hide(axis='index'))

Tolerance τ (°C),Pairs mergeable,% of category A,Pairs kept separate,Compounds after merging
0,"25,638",97.1,774,"278,870"
2,"26,275",99.5,137,"278,233"
5,"26,329",99.7,83,"278,179"
10,"26,353",99.8,59,"278,155"


## Largest disagreements

For unspecified–specified pairs, `a` is the unspecified entry and `b` the specified one.

In [8]:
display(pairs.sort_values('Abs_dMP', ascending=False).head(10)[
    ['Category', 'Pair_type', 'SMILES_a', 'MP_a', 'Source_a', 'Quality_a',
     'SMILES_b', 'MP_b', 'Source_b', 'Quality_b', 'Abs_dMP']
].reset_index(drop=True))

,Category,Pair_type,SMILES_a,MP_a,Source_a,Quality_a,SMILES_b,MP_b,Source_b,Quality_b,Abs_dMP
0,B: 1 unspecified + ≥2 specified,specified–specified,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@@H](Cl)[C@H](Cl)[C@H]1Cl,112.50,4,single,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@@H]1Cl,314.5,4,single,202.00
1,B: 1 unspecified + ≥2 specified,specified–specified,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@@H]1Cl,314.50,4,single,Cl[C@H]1[C@H](Cl)[C@H](Cl)[C@H](Cl)[C@@H](Cl)[C@H]1Cl,112.5,4,single,202.00
2,B: 1 unspecified + ≥2 specified,unspecified–specified,ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,114.00,2+4,exact,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@@H]1Cl,314.5,4,single,200.50
3,A: 1 unspecified + 1 specified,unspecified–specified,CCOC(=O)C(C#N)=Cc1ccc(N(C)C)cc1,311.00,3,single,CCOC(=O)/C(C#N)=C\c1ccc(N(C)C)cc1,122.5,4,single,188.50
4,A: 1 unspecified + 1 specified,unspecified–specified,Clc1ccc(C=NNc2ccccc2)cc1,311.00,3,single,Clc1ccc(/C=N/Nc2ccccc2)cc1,127.0,4,single,184.00
5,B: 1 unspecified + ≥2 specified,specified–specified,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@@H]1Cl,314.50,4,single,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@H]1Cl,141.5,4,single,173.00
6,B: 1 unspecified + ≥2 specified,specified–specified,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@@H](Cl)[C@@H]1Cl,314.50,4,single,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@H](Cl)[C@H](Cl)[C@H]1Cl,159.1,3+4,near_exact,155.40
7,B: 1 unspecified + ≥2 specified,specified–specified,COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)C[C@@](O)(C(=O)CO)C[C@@H]3O[C@H]1C[C@H](N)[C@@H](...,344.53,3+4,exact,COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)C[C@@](O)(C(=O)CO)C[C@@H]3O[C@H]1C[C@H](N)[C@H](O...,204.5,4,single,140.03
8,C: specified only,specified–specified,C(=C/c1ccccc1)\c1ccccc1,124.25,3+4,near_exact,C(=C\c1ccccc1)\c1ccccc1,2.0,4,single,122.25
9,B: 1 unspecified + ≥2 specified,specified–specified,COC(=O)c1ccc(/C=C/c2ccc(C(=O)OC)cc2)cc1,235.00,3+4,exact,COC(=O)c1ccc(/C=C\c2ccc(C(=O)OC)cc2)cc1,113.0,4,single,122.00


## Category D profile

Category D compounds carry stereo labels but have no unlabelled version and no alternative stereoisomer in the dataset. Stereo completeness uses `Chem.FindPotentialStereo`: **fully specified** when every potential stereocentre and stereo double bond carries a mark, **partially specified** otherwise.

In [9]:
def stereo_completeness(smiles):
    stereo_info = Chem.FindPotentialStereo(Chem.MolFromSmiles(smiles))
    n_specified = sum(item.specified == Chem.StereoSpecified.Specified for item in stereo_info)
    if n_specified == 0:
        return 'unspecified'
    return 'fully specified' if n_specified == len(stereo_info) else 'partially specified'

category_d = combined_data.loc[labelled & ~grouped].copy()
category_d['Completeness'] = category_d['SMILES_isomeric'].map(stereo_completeness)
category_d['Sources'] = category_d['Source_label'].map(source_names)
category_d['MP_class'] = np.where(category_d['MP'].ge(HIGH_THRESHOLD),
                                  'High (≥250 °C)', 'Low (<250 °C)')

profile = pd.concat([
    category_d[column].value_counts().rename('Compounds').to_frame().assign(Attribute=label)
    for column, label in [
        ('Stereo_element', 'Stereo element'), ('Completeness', 'Stereo completeness'),
        ('Sources', 'Sources'), ('MP_class', 'MP class'), ('MP_quality', 'MP quality'),
    ]
]).rename_axis('Value').reset_index().set_index(['Attribute', 'Value'])
profile['% of category D'] = 100 * profile['Compounds'] / len(category_d)
print(f'Category D: {len(category_d):,} compounds')
display(profile.style.format({'Compounds': '{:,}', '% of category D': '{:.1f}'}))

Category D: 4,736 compounds


# Part 2: apply the stereo clean-up

The agreed rule, applied to `combined_data_before_stereo_cleanup.parquet`:

- **Category A** (1 unspecified + 1 specified compound): one entry is kept. The PATENTS entry is preferred; if neither entry comes from PATENTS, the stereo-specified entry is kept. Pairs whose MPs differ by more than `STEREO_CONFLICT_TOLERANCE_C` (10 °C, the agreement threshold used for the cross-source consensus) are removed entirely. The kept row is unchanged.
- **Categories B and C:** removed entirely.
- **Category D and all other compounds:** unchanged.

After the clean-up every non-isomeric SMILES occurs once, so no stereo duplicates remain and the standard stratified split can be used. `stereo_cleanup_audit.csv` lists every group member and the action taken.

In [10]:
def write_combined_parquet(frame, path):
    table = pa.Table.from_pandas(frame, preserve_index=False)
    source_index = table.column_names.index('Source')
    table = table.set_column(
        source_index, 'Source',
        pa.array(frame['Source'].tolist(), type=pa.list_(pa.int8()))
    )
    pq.write_table(table, path, compression='zstd')

# nonisomeric_smiles() and stereo_stripped_smiles() are defined in Part 1.
before_cleanup = pd.read_parquet(BEFORE_CLEANUP_PATH)
# Parquet returns Source as int8 arrays; convert back to plain integer lists, as written by
# 1_combine_data.ipynb, so the saved file is identical to writing the table directly.
before_cleanup['Source'] = before_cleanup['Source'].map(lambda values: [int(value) for value in values])
stereo_frame = before_cleanup[['SMILES', 'MP', 'Source']].copy()
stereo_frame['SMILES_nonisomeric'] = stereo_frame['SMILES'].map(nonisomeric_smiles)
stereo_frame['Has_stereo'] = stereo_frame['SMILES'].str.contains(STEREO_MARKS)
group_size = stereo_frame.groupby('SMILES_nonisomeric')['SMILES'].transform('size')
members = stereo_frame.loc[group_size.gt(1)].copy()

# Group members may differ only in stereochemistry, not, for example, in isotope labels.
members['SMILES_stereo_stripped'] = members['SMILES'].map(stereo_stripped_smiles)
if members.groupby('SMILES_nonisomeric')['SMILES_stereo_stripped'].nunique().gt(1).any():
    raise ValueError('Some stereo groups differ by more than stereochemistry.')

composition = members.groupby('SMILES_nonisomeric').agg(
    N_members=('SMILES', 'size'), N_specified=('Has_stereo', 'sum'),
)
composition['N_unspecified'] = composition['N_members'] - composition['N_specified']
composition['Category'] = np.select(
    [composition['N_unspecified'].eq(1) & composition['N_specified'].eq(1),
     composition['N_unspecified'].eq(1) & composition['N_specified'].ge(2),
     composition['N_unspecified'].eq(0)],
    ['A', 'B', 'C'], 'other',
)
if composition['Category'].eq('other').any():
    raise ValueError('Unexpected stereo-group composition.')
members = members.join(composition['Category'], on='SMILES_nonisomeric')
members['From_PATENTS'] = members['Source'].map(lambda sources: 3 in set(sources))
members['MP_spread_in_group'] = members.groupby('SMILES_nonisomeric')['MP'].transform(
    lambda values: values.max() - values.min()
)

category_a = members['Category'].eq('A')
patents_entries = members.loc[category_a].groupby('SMILES_nonisomeric')['From_PATENTS'].sum()
if patents_entries.gt(1).any():
    raise ValueError('A category-A pair has PATENTS in both entries; the preference rule is ambiguous.')
group_has_patents = members['SMILES_nonisomeric'].map(patents_entries).fillna(0).gt(0)
conflict = category_a & members['MP_spread_in_group'].gt(STEREO_CONFLICT_TOLERANCE_C)
preferred = members['From_PATENTS'].where(group_has_patents, members['Has_stereo']).astype(bool)
members['Action'] = np.select(
    [members['Category'].eq('B'), members['Category'].eq('C'), conflict, category_a & preferred],
    ['removed_category_B', 'removed_category_C', 'removed_conflict', 'kept'],
    'removed_duplicate',
)
kept_per_pair = members.loc[category_a & ~conflict].groupby('SMILES_nonisomeric')['Action'].agg(
    lambda actions: int((actions == 'kept').sum())
)
if not kept_per_pair.eq(1).all():
    raise ValueError('Every retained category-A pair must keep exactly one entry.')

stereo_audit = members[[
    'SMILES_nonisomeric', 'Category', 'SMILES', 'Has_stereo', 'MP', 'Source',
    'From_PATENTS', 'MP_spread_in_group', 'Action',
]].copy()
stereo_audit['Source'] = stereo_audit['Source'].map(lambda sources: '+'.join(map(str, sorted(sources))))
stereo_audit = stereo_audit.sort_values(['Category', 'SMILES_nonisomeric', 'SMILES'], kind='stable')
stereo_audit.to_csv(OUTPUT_STEREO_AUDIT, index=False)

removed_smiles = set(members.loc[members['Action'].ne('kept'), 'SMILES'])
combined_data = before_cleanup.loc[~before_cleanup['SMILES'].isin(removed_smiles)].reset_index(drop=True)
write_combined_parquet(combined_data, OUTPUT_DATA)

cleanup_summary = (
    stereo_audit.groupby(['Category', 'Action']).agg(
        Groups=('SMILES_nonisomeric', 'nunique'), Compounds=('SMILES', 'size'),
    ).reset_index()
)
display(cleanup_summary)
print(f'Compounds before the stereo clean-up: {len(before_cleanup):,}')
print(f'Compounds removed: {len(removed_smiles):,}')
print(f'Final compounds: {len(combined_data):,}')
print(f'Wrote: {OUTPUT_DATA.relative_to(DATA_DIR.parent)}')
print(f'Wrote: {OUTPUT_STEREO_AUDIT.relative_to(DATA_DIR.parent)}')

,Category,Action,Groups,Compounds
0,A,kept,26353,26353
1,A,removed_conflict,59,118
2,A,removed_duplicate,26353,26353
3,B,removed_category_B,250,767
4,C,removed_category_C,66,136


Compounds before the stereo clean-up: 304,508
Compounds removed: 27,374
Final compounds: 277,134
Wrote: 0_data/outputs/combined_data.parquet
Wrote: 0_data/outputs/stereo_cleanup_audit.csv


## Validate the final dataset

In [11]:
saved_table = pq.read_table(OUTPUT_DATA)
saved = saved_table.to_pandas()
saved_stereo_audit = pd.read_csv(OUTPUT_STEREO_AUDIT)
removed = set(saved_stereo_audit.loc[saved_stereo_audit['Action'].ne('kept'), 'SMILES'])

assert saved.columns.tolist() == before_cleanup.columns.tolist()
assert saved_table.schema.field('Source').type == pq.read_schema(BEFORE_CLEANUP_PATH).field('Source').type
assert saved['SMILES'].is_unique
assert set(saved['SMILES']) == set(before_cleanup['SMILES']) - removed
assert len(saved) == len(before_cleanup) - len(removed)
assert saved['SMILES'].map(nonisomeric_smiles).is_unique, 'Stereo duplicates remain after the clean-up.'
print(f'Final compounds: {len(saved):,}')
print('All validation checks passed.')

Final compounds: 277,134
All validation checks passed.


## Graph embeddings for the final dataset

The graph embeddings in `input_datasets/graph_embeddings/combined_data_SingleViews_graph_embeddings.csv.gz` were precomputed by an external graph model for every compound of `combined_data_before_stereo_cleanup.parquet`. They are not regenerated here. The rows of compounds removed by the stereo clean-up are dropped, and the remaining rows are copied byte-for-byte to `combined_data_graph_embeddings.csv.gz`, so that the embedding file contains exactly one row per compound of `combined_data.parquet`. The source file is only read.

In [12]:
final_smiles = set(combined_data['SMILES'])
written_smiles = set()
kept_rows = dropped_rows = 0
with gzip.open(GRAPH_EMBEDDINGS_SOURCE, 'rb') as source_handle, \
     gzip.GzipFile(OUTPUT_GRAPH_EMBEDDINGS, 'wb', compresslevel=6, mtime=0) as target_handle:
    header = source_handle.readline()
    assert header.rstrip(b'\r\n') == b'smiles,graph_emb', header
    target_handle.write(header)
    for line in source_handle:
        smiles = line.split(b',', 1)[0].decode()
        assert not smiles.startswith('"'), smiles
        if smiles in final_smiles:
            assert smiles not in written_smiles, f'Duplicate graph embedding for {smiles}'
            written_smiles.add(smiles)
            target_handle.write(line)
            kept_rows += 1
        else:
            dropped_rows += 1

assert written_smiles == final_smiles, f'{len(final_smiles - written_smiles):,} compounds have no graph embedding.'
assert kept_rows == len(combined_data)
display(pd.DataFrame({
    'Rows': ['Source embeddings', 'Kept (final dataset)', 'Dropped (removed compounds)'],
    'Count': [kept_rows + dropped_rows, kept_rows, dropped_rows],
}))
print(f'Wrote: {OUTPUT_GRAPH_EMBEDDINGS}')

,Rows,Count
0,Source embeddings,304508
1,Kept (final dataset),277134
2,Dropped (removed compounds),27374


Wrote: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/outputs/combined_data_graph_embeddings.csv.gz
